In [1]:
!pip install transformers datasets torch pandas scikit-learn
!pip install imbalanced-learn

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import tensorflow as tf



from transformers import BertTokenizer, BertForSequenceClassification

from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

2026-03-15 04:47:58.192518: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773550078.384022      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773550078.442739      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773550078.925817      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773550078.925858      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773550078.925861      55 computation_placer.cc:177] computation placer alr

# Uploding dataset

In [3]:
DATA_PATH = "//kaggle/input/datasets/mrsikandarali/metahate-dataset/your_dataset_updated.tsv"

df = pd.read_csv(DATA_PATH, sep="\t")

print(df.head())
print("Dataset shape:", df.shape)

   label                                               text
0      0  !!! RT @mayasolovely: As a woman you shouldn't...
1      0  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2      0  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3      0  !!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4      0  !!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...
Dataset shape: (1101165, 2)


# Checking dataset

In [4]:
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True))

label
0    867876
1    233289
Name: count, dtype: int64
label
0    0.788143
1    0.211857
Name: proportion, dtype: float64


# Selecting same data numbers

In [5]:
# Selecting same data numbers
hate = df[df.label == 1]
non_hate = df[df.label == 0].sample(len(hate), random_state=42)

balanced_df = pd.concat([hate, non_hate])

# show the number of balanced dataset
balanced_df['label'].value_counts()

label
1    233289
0    233289
Name: count, dtype: int64

In [6]:
balanced_df.to_csv("balanced_dataset.csv", index=False)

In [7]:
import os
os.listdir("/kaggle/working")

balanced_df.to_csv("/kaggle/working/balanced_dataset.csv", index=False)

# knwo we will start the testing on same code

In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/mrsikandarali/balanced-dataset/balanced_dataset.csv")

print(df.head())

   label                                               text
0      1  "@Blackman38Tide: @WhaleLookyHere @HowdyDowdy1...
1      1  "@CB_Baby24: @white_thunduh alsarabsss" hes a ...
2      1  "@DevilGrimz: @VigxRArts you're fucking gay, b...
3      1  "@MarkRoundtreeJr: LMFAOOOO I HATE BLACK PEOPL...
4      1  "@NoChillPaz: "At least I'm not a nigger" http...


In [3]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [4]:
# Detect hardware, return appropriate distribution strategy
# Step 1: Import TensorFlow

try:
    # TPU detection
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Running on TPU:', tpu.master())
except ValueError:
    tpu = None

if tpu:
    # Connect to TPU
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
else:
    # Default strategy for CPU and GPU
    strategy = tf.distribute.get_strategy()

print("REPLICAS:", strategy.num_replicas_in_sync)

# Check available devices

print("GPU Available:", tf.config.list_physical_devices('GPU') != [])
print("CPU Available:", tf.config.list_physical_devices('CPU') != [])
print("TPU Available:", tf.config.list_physical_devices('TPU') != [])

REPLICAS: 1
GPU Available: True
CPU Available: True
TPU Available: False


In [5]:
# Load fast tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

MAX_LEN = 128


def tokenize_batch(texts, batch_size=50000, max_length=MAX_LEN):

    encodings = {
        "input_ids": [],
        "attention_mask": [],
        "token_type_ids": []
    }

    total = len(texts)

    for i in range(0, total, batch_size):

        batch = texts[i:i+batch_size].tolist()

        tokens = tokenizer(
            batch,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        encodings["input_ids"].extend(tokens["input_ids"])
        encodings["attention_mask"].extend(tokens["attention_mask"])
        encodings["token_type_ids"].extend(tokens["token_type_ids"])

        print(f"Processed {min(i+batch_size, total)} / {total}")

    return encodings


# Tokenize datasets
train_encodings = tokenize_batch(train_texts)
test_encodings = tokenize_batch(test_texts)


# Save encoded dataset
torch.save({
    "train_encodings": train_encodings,
    "test_encodings": test_encodings,
    "train_labels": train_labels,
    "test_labels": test_labels
}, "dataset_encodings.pt")


# Save tokenizer
tokenizer.save_pretrained("saved_tokenizer")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Processed 50000 / 373262
Processed 100000 / 373262
Processed 150000 / 373262
Processed 200000 / 373262
Processed 250000 / 373262
Processed 300000 / 373262
Processed 350000 / 373262
Processed 373262 / 373262
Processed 50000 / 93316
Processed 93316 / 93316


('saved_tokenizer/tokenizer_config.json', 'saved_tokenizer/tokenizer.json')

In [6]:
# Load dataset
data = torch.load("dataset_encodings.pt", map_location="cpu", weights_only=False)

# Extract items
train_encodings = data["train_encodings"]
test_encodings = data["test_encodings"]
train_labels = data["train_labels"]
test_labels = data["test_labels"]

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("saved_tokenizer")

print("Dataset loaded successfully")

Dataset loaded successfully


### Create PyTorch Dataset Class

In [7]:
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Use iloc to get by position
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

# Create Train and Test Dataset

In [8]:
train_dataset = HateDataset(train_encodings, train_labels)
test_dataset = HateDataset(test_encodings, test_labels)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Load BERT Model for Classification

In [10]:
from transformers import BertForSequenceClassification
import torch

# Load BERT for sequence classification with 2 labels
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# Choose device: GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the device
model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [11]:
!pip install tqdm 
from tqdm import tqdm
optimizer = AdamW(model.parameters(), lr=2e-5)

In [13]:
#### from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score

# Mixed precision scaler (speed improvement)
scaler = GradScaler()

epochs = 5
patience = 1        # Early stopping patience
best_val_loss = float("inf")
counter = 0

for epoch in range(epochs):

    # ========================
    # TRAINING
    # ========================
    model.train()
    total_loss = 0

    print(f"\nStarting Epoch {epoch+1}/{epochs}")

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Mixed precision (faster training)
        with autocast():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        progress_bar.set_postfix({"Batch Loss": loss.item()})

    avg_train_loss = total_loss / len(train_loader)

    # ========================
    # VALIDATION
    # ========================
    model.eval()

    val_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(test_loader)
    accuracy = accuracy_score(true_labels, predictions)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.4f}")

    # ========================
    # EARLY STOPPING
    # ========================
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss
        counter = 0

        torch.save(model.state_dict(), "best_bert_model.pt")

        print("Model improved. Saved.")

    else:

        counter += 1
        print(f"No improvement. Early stop counter: {counter}/{patience}")

        if counter >= patience:

            print("Early stopping triggered.")
            break

/tmp/ipykernel_55/228433217.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Starting Epoch 1/5


Epoch 1:   0%|          | 0/11665 [00:00<?, ?it/s]/tmp/ipykernel_55/228433217.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1: 100%|██████████| 11665/11665 [39:04<00:00,  4.98it/s, Batch Loss=0.615] 


Epoch 1
Train Loss: 0.3247
Validation Loss: 0.2983
Validation Accuracy: 0.8683
Model improved. Saved.

Starting Epoch 2/5


Epoch 2:   0%|          | 0/11665 [00:00<?, ?it/s]/tmp/ipykernel_55/228433217.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 2: 100%|██████████| 11665/11665 [39:07<00:00,  4.97it/s, Batch Loss=0.522] 


Epoch 2
Train Loss: 0.2540
Validation Loss: 0.2876
Validation Accuracy: 0.8778
Model improved. Saved.

Starting Epoch 3/5


Epoch 3:   0%|          | 0/11665 [00:00<?, ?it/s]/tmp/ipykernel_55/228433217.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 3: 100%|██████████| 11665/11665 [39:13<00:00,  4.96it/s, Batch Loss=0.0787]


Epoch 3
Train Loss: 0.1913
Validation Loss: 0.3247
Validation Accuracy: 0.8766
No improvement. Early stop counter: 1/1
Early stopping triggered.


In [14]:
# # Save
model.save_pretrained("./best_bert_model")
tokenizer.save_pretrained("./best_bert_model")  # save tokenizer too
print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


In [15]:
# Load later
from transformers import BertForSequenceClassification, BertTokenizer

model = BertForSequenceClassification.from_pretrained("./best_bert_model")
tokenizer = BertTokenizer.from_pretrained("./best_bert_model")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [16]:
# model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    
    for batch in test_loader:
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

In [17]:
accuracy = accuracy_score(true_labels, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(true_labels, predictions))

Accuracy: 0.8765806506922714

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.85      0.87     46743
           1       0.86      0.90      0.88     46573

    accuracy                           0.88     93316
   macro avg       0.88      0.88      0.88     93316
weighted avg       0.88      0.88      0.88     93316



In [18]:
def predict(text):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    outputs = model(**inputs)
    
    prediction = torch.argmax(outputs.logits, dim=1).item()
    
    return prediction


example = "I hate this community"

print("Prediction:", predict(example))

Prediction: 0
